In [1]:
import numpy as np
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
import torch
from matplotlib import pyplot as plt
from jordanutils import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import pandas as pd
from collections import defaultdict

rng = np.random.default_rng(seed=123)

In [2]:
MODES = ['random', 'int', 'upper', 'lower', 'ortho', 'random_eps']

In [3]:
def setup_device():
    try:
        import torch_directml

        device = torch_directml.device()
        backend = "directml"    
    except ImportError:
        if torch.cuda.is_available():
            device = torch.device("cuda")
            backend = "cuda"
        else:
            device = torch.device("cpu")
            backend = "cpu"
    return device


## Training on all "signle" datasets 

In [4]:
# ---- Define the PyTorch model ----
class SimpleNN(nn.Module):
    def __init__(self, d):
        super(SimpleNN, self).__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(d * d, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, d),
        )

    def forward(self, x):
        x = self.flatten(x)
        x = self.layers(x)
        return x

# ---- Training + Evaluation function ----
def generate_and_train_model(X, y, device, verbose=1, epochs=50):
    # --- Dataset setup ---
    d = X[0].shape[0]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=21
    )

    # --- Convert to tensors ---
    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.long)

    # --- DataLoaders ---
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

    # --- Model, Loss, Optimizer ---
    model = SimpleNN(d).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # --- Early Stopping setup ---
    best_val_acc = 0.0
    patience = 5
    counter = 0
    best_weights = None

    for epoch in range(epochs):
        model.train()
        # Initialize running loss for the epoch
        running_loss = 0.0
        counter = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            # Accumulate training loss
            running_loss += loss.item() * xb.size(0)
            counter += yb.size(0)

        # Calculate average training loss for the epoch
        train_loss = running_loss / counter

        # --- Validation Loop ---
        model.eval()
        correct, total = 0, 0
        # Initialize running validation loss
        val_running_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                preds = model(xb)
                # Calculate validation loss
                val_loss = criterion(preds, yb)
                val_running_loss += val_loss.item() * xb.size(0)

                pred_labels = preds.argmax(1)
                correct += (pred_labels == yb).sum().item()
                total += yb.size(0)
        
        # Calculate average validation loss for the epoch
        avg_val_loss = val_running_loss / total
        val_acc = correct / total

        if verbose:
            print(f"Epoch {epoch+1:02d} - Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # --- Early Stopping ---
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            counter = 0
            best_weights = model.state_dict()
        else:
            counter += 1
            if epoch == 10:
                counter = 0
            if counter >= patience and epoch > 10:
                if verbose:
                    print(f"Early stopping at epoch {epoch+1}")
                break

    # --- Restore best model ---
    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model



In [5]:
device = setup_device()
size_per_class = 50000
d = 5
# X, y = generate_testset(d, size_per_class=size_per_class, mode='random')
# model = generate_and_train_model(X, y, device)

In [6]:
testsets = {}
for mode in MODES:
    _test_mode, _eps_test = ("random", 0.01) if mode == 'random_eps' else (mode, None)
    testsets[mode] = generate_testset(d, size_per_class=1000, mode=_test_mode, eps=_eps_test)

In [7]:
def run_testcase(model, X_test):    
    X_test = torch.tensor(X_test, dtype=torch.float32).to(device)
    model.eval()

    with torch.no_grad():
        outputs = model(X_test)
        y_predicted = torch.argmax(outputs, dim=1).cpu().numpy()

    return y_predicted

In [8]:
def run_and_gather_results(testsets):
    train_modes = MODES
    test_modes = MODES
    full_results = defaultdict(dict)
    accuracy_results = defaultdict(dict)
    for train_mode in train_modes: 
        _train_mode, _eps_train = ("random", 0.01) if train_mode == 'random_eps' else (train_mode, None)
        X_train, y_train = generate_testset(d, 50_000, _train_mode, _eps_train)
        model = generate_and_train_model(X_train, y_train, device, epochs=50)
        torch.save(model.state_dict(), f"model_{train_mode}.pth")
        for test_mode in test_modes:
            X_test, y_test = testsets[test_mode]
            y_pred = run_testcase(model, X_test)
            accuracy = accuracy_score(y_test, y_pred)
            cm = confusion_matrix(y_test, y_pred)

            full_results[test_mode][train_mode] = cm
            accuracy_results[test_mode][train_mode] = accuracy

            print(f" ###### Train: {train_mode} | Test: {test_mode} | Accuracy: {accuracy:.4f} ######")

    return full_results, accuracy_results

In [9]:
full_results, accuracy_results = run_and_gather_results(testsets)

c:\Users\micha\Documents\Studia\Magisterka\venv\Lib\site-packages\torch\optim\adam.py:534: UserWarning: The operator 'aten::lerp.Scalar_out' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  torch._foreach_lerp_(device_exp_avgs, device_grads, 1 - beta1)


Epoch 01 - Train Loss: 1.0429 | Val Loss: 1.0094 | Val Acc: 0.4994
Epoch 02 - Train Loss: 1.0025 | Val Loss: 0.9968 | Val Acc: 0.5073
Epoch 03 - Train Loss: 0.9775 | Val Loss: 0.9482 | Val Acc: 0.5408
Epoch 04 - Train Loss: 0.8988 | Val Loss: 0.8876 | Val Acc: 0.5717
Epoch 05 - Train Loss: 0.8382 | Val Loss: 0.8406 | Val Acc: 0.5930
Epoch 06 - Train Loss: 0.8108 | Val Loss: 0.8191 | Val Acc: 0.6017
Epoch 07 - Train Loss: 0.7888 | Val Loss: 0.8018 | Val Acc: 0.6143
Epoch 08 - Train Loss: 0.7710 | Val Loss: 0.7951 | Val Acc: 0.6155
Epoch 09 - Train Loss: 0.7678 | Val Loss: 0.7826 | Val Acc: 0.6225
Epoch 10 - Train Loss: 0.7483 | Val Loss: 0.7760 | Val Acc: 0.6257
Epoch 11 - Train Loss: 0.7418 | Val Loss: 0.7787 | Val Acc: 0.6259
Epoch 12 - Train Loss: 0.7318 | Val Loss: 0.7664 | Val Acc: 0.6329
Epoch 13 - Train Loss: 0.7220 | Val Loss: 0.7579 | Val Acc: 0.6365
Epoch 14 - Train Loss: 0.7147 | Val Loss: 0.7701 | Val Acc: 0.6303
Early stopping at epoch 14
 ###### Train: random | Test: rando

In [10]:
def save_data(data_to_save, filename):
    flat_data = {}
    for outer_key, inner_dict in data_to_save.items():
        for inner_key, array in inner_dict.items():
            flat_data[f"test:{outer_key}/train:{inner_key}"] = array

    # 2. Save the flattened data
    np.savez_compressed(f'{filename}.npz', **flat_data)
    print(f"Data saved to {filename}.npz")

In [11]:
save_data(accuracy_results, 'accuracy_results')
save_data(full_results, 'full_results')

Data saved to accuracy_results.npz
Data saved to full_results.npz


In [12]:
print(accuracy_results)

defaultdict(<class 'dict'>, {'random': {'random': 0.6248, 'int': 0.482, 'upper': 0.403, 'lower': 0.4096, 'ortho': 0.4266, 'random_eps': 0.6134}, 'int': {'random': 0.5776, 'int': 0.6916, 'upper': 0.4036, 'lower': 0.4068, 'ortho': 0.4026, 'random_eps': 0.5808}, 'upper': {'random': 0.5712, 'int': 0.465, 'upper': 0.89, 'lower': 0.6746, 'ortho': 0.43, 'random_eps': 0.5446}, 'lower': {'random': 0.5198, 'int': 0.455, 'upper': 0.4884, 'lower': 0.878, 'ortho': 0.409, 'random_eps': 0.5096}, 'ortho': {'random': 0.8076, 'int': 0.526, 'upper': 0.4194, 'lower': 0.4574, 'ortho': 1.0, 'random_eps': 0.8758}, 'random_eps': {'random': 0.5128, 'int': 0.4294, 'upper': 0.2824, 'lower': 0.241, 'ortho': 0.3814, 'random_eps': 0.5958}})


### Training on `full` dataset

In [15]:
def run_full_training(testsets):
    X_list = []
    y_list = []
    for mode in MODES:
        _mode, _eps = ("random", 0.01) if mode == 'random_eps' else (mode, None)
        X, y = generate_testset(5, 50_000, mode=_mode, eps=_eps)
        X_list.append(X)
        y_list.append(y)
    X_train = np.concat(X_list, axis=0)
    y_train = np.concat(y_list, axis=0)
    model = generate_and_train_model(X_train, y_train, device, epochs=50)
    torch.save(model.state_dict(), "model_full.pth")
    accuracy_results = defaultdict(dict)
    full_results = defaultdict(dict)
    for test_mode in MODES:
        X_test, y_test = testsets[test_mode]
        y_pred = run_testcase(model, X_test)
        accuracy = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)

        full_results[test_mode]["full"] = cm
        accuracy_results[test_mode]["full"] = accuracy

        print(f" ###### Train: full | Test: {test_mode} | Accuracy: {accuracy:.4f} ######")

    return full_results, accuracy_results

In [16]:
full_results, accuracy_results = run_full_training(testsets)

save_data(accuracy_results, 'accuracy_results')
save_data(full_results, 'full_results')

Epoch 01 - Train Loss: 0.8637 | Val Loss: 0.7582 | Val Acc: 0.6388
Epoch 02 - Train Loss: 0.7256 | Val Loss: 0.7070 | Val Acc: 0.6636
Epoch 03 - Train Loss: 0.6813 | Val Loss: 0.6618 | Val Acc: 0.6885
Epoch 04 - Train Loss: 0.6539 | Val Loss: 0.6515 | Val Acc: 0.6929
Epoch 05 - Train Loss: 0.6372 | Val Loss: 0.6411 | Val Acc: 0.6990
Epoch 06 - Train Loss: 0.6538 | Val Loss: 0.6712 | Val Acc: 0.7105
Epoch 07 - Train Loss: 0.6148 | Val Loss: 0.6083 | Val Acc: 0.7154
Epoch 08 - Train Loss: 0.6027 | Val Loss: 0.6109 | Val Acc: 0.7153
Epoch 09 - Train Loss: 0.5923 | Val Loss: 0.5810 | Val Acc: 0.7287
Epoch 10 - Train Loss: 0.8372 | Val Loss: 0.5759 | Val Acc: 0.7329
Epoch 11 - Train Loss: 0.6003 | Val Loss: 0.5786 | Val Acc: 0.7320
Epoch 12 - Train Loss: 0.5808 | Val Loss: 0.5623 | Val Acc: 0.7385
Epoch 13 - Train Loss: 0.5605 | Val Loss: 0.5612 | Val Acc: 0.7394
Epoch 14 - Train Loss: 0.5407 | Val Loss: 0.5442 | Val Acc: 0.7499
Epoch 15 - Train Loss: 0.6411 | Val Loss: 0.5244 | Val Acc: 0.